In [1]:
pip install mediapipe

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\cmedi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pygame
import time

pygame 2.6.1 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
pygame.mixer.init()

try:
    sound = pygame.mixer.Sound("../media/ding.wav")
except:
    sound = None

In [4]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [ ]:
cap = cv2.VideoCapture(0)

last_action = ""

left_foot_history = []
right_foot_history = []

def play_sound():
    if sound:
        sound.play()

In [ ]:
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = pose.process(rgb)

    action = "Sin detectar"

    if results.pose_landmarks:

        landmarks = results.pose_landmarks.landmark

        nose = landmarks[mp_pose.PoseLandmark.NOSE]

        left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST]
        right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST]

        left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP]
        right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP]

        left_knee = landmarks[mp_pose.PoseLandmark.LEFT_KNEE]
        right_knee = landmarks[mp_pose.PoseLandmark.RIGHT_KNEE]

        left_ankle = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE]
        right_ankle = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE]


        if (
            left_wrist.y < nose.y and
            right_wrist.y < nose.y
        ):
            action = "Brazos levantados"


        elif (
            left_hip.y > left_knee.y and
            right_hip.y > right_knee.y
        ):
            action = "Sentado"

        left_foot_history.append(left_ankle.y)
        right_foot_history.append(right_ankle.y)

        if len(left_foot_history) > 10:
            left_foot_history.pop(0)

        if len(right_foot_history) > 10:
            right_foot_history.pop(0)

        if len(left_foot_history) == 10:

            left_variation = max(left_foot_history) - min(left_foot_history)
            right_variation = max(right_foot_history) - min(right_foot_history)

            if left_variation > 0.03 and right_variation > 0.03:
                action = "Caminando"


        if action != last_action:
            play_sound()
            last_action = action


        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

    cv2.putText(
        frame,
        f"Accion: {action}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow("Reconocimiento de Postura", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break



C:\Users\cmedi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

: 

In [ ]:
cap.release()
cv2.destroyAllWindows()